# `data/` 資料模組使用手冊

## 1. 本資料夾有哪些資料路徑？

`data/` 同時放著 smoke 測試資料、目前 KD 訓練 Dataset、原始資料下載器和一個舊格式 loader。它們不是同一條 pipeline，使用時要看清楚用途：

| 檔案 | 用途 | 目前是否由正式 `train_kd.py` 使用 |
|---|---|---|
| `synthetic_smoke_dataset.py` | 產生隨機 RGB、placeholder sigma 和 toy target，檢查 KD plumbing | 否，只由 `run_kd_smoke.py` 使用 |
| `download_open_images.py` | 用 FiftyOne 下載 Open Images V6 train 圖片 | 作為訓練前下載步驟執行 |
| `nafnet_kd_dataset.py` | 載入 clean image、抽 crop、建立 burst、加噪聲、計算 sigma map | 是 |
| `verify_kd_dataset.py` | 用小量 source images 檢查生成結果和 shape | 人工驗證工具，不進入 training loop |
| `burst_dataset.py` | 從 `.npz` 讀取 pre-generated burst 和參數 | 否；舊的獨立 loader |

**Smoke data 與 KD training data 不可混稱「真實資料」。**Smoke Dataset 的 target 是隨機張量、sigma map 是固定常數，只能驗證維度和反向傳播；正式 KD Dataset 的來源是實際下載的 clean photographs，再由程式合成 noisy burst。

## 2. 目前正式 KD 資料流

```text
Open Images V6 原始 RGB photograph
  │
  ├─ 限制：來源圖寬、高至少 640 pixel
  ├─ 取 640×640 區域
  ├─ 產生 8 個 128×128 對齊 frames（frame 0 為 reference）
  ├─ 套用 gamma / white-level 縮放
  ├─ sample read noise、shot noise；得到 noisy burst
  ├─ 以同一組 noise parameters 計算每幀、每通道 sigma map
  │
  ├─ noisy RGB burst + sigma map → Teacher input [8, 6, 128, 128]
  ├─ noisy RGB frame 0         → Student input [3, 128, 128]
  └─ clean RGB frame 0         → clean target [3, 128, 128]
```

Teacher 和 Student 看到的是同一個樣本中的 reference：Student 使用 noisy frame 0；Teacher 接收包含該 frame 在內的 8 frames。clean target 是對應 frame 0 的 clean、縮放後影像。這三者在同一次 `__getitem__` 生成，避免把不同影像的 noisy/clean pair 錯配。

資料是**即時生成（on-the-fly）**。目前不會把 8 frames noisy burst、clean target 或 sigma map 寫到 `data/processed/`；硬碟上只保存 FiftyOne 下載的 source photographs 和 metadata。`processed/` 目前是預留位置，沒有被 trainer 使用。

## 3. 資料夾管理與 Open Images 下載

```text
data/
├── raw/
│   └── open-images-v6/
│       └── train/
│           ├── data/                 # 直接放下載的 clean source images
│           └── metadata/             # FiftyOne 取得的 dataset metadata
├── processed/                        # 預留，目前沒有訓練輸出放在這裡
├── download_open_images.py
├── verify_kd_dataset.py
├── nafnet_kd_dataset.py
├── synthetic_smoke_dataset.py
├── burst_dataset.py
└── README.ipynb
```

在專案根目錄（不是 `data/` 子目錄）執行：

```powershell
python -m data.download_open_images --max-samples 10
```

Downloader 使用 FiftyOne 的 Open Images V6 train split，設定 `label_types=[]`，不需要物件標註，只取圖片。`--max-samples` 表示最多下載的**來源圖片數**；預設為 10，方便先做小型測試。正式擴量時可指定上限，例如 `--max-samples 20000`，但此數字本身不保證有同樣多的可用圖或 training patches。

Downloader 將內容放在 `data/raw/open-images-v6/train/`，其中 `data/` 放圖片，`metadata/` 放 FiftyOne 的索引資訊。下載需要網路，metadata 可能相當大；首次下載預計時間依連線和 FiftyOne cache 狀態而變。若程式回報 `Necessary images already downloaded`，代表它重用既有資料，不表示又下載了一份。

目前 Dataset 接受 `.jpg`、`.jpeg`、`.png`，只掃描指定 `image_dir` 的**直接子檔案**，不遞迴搜尋。不能讀取的圖片及短邊不足 640 pixel 的圖片會被略過。小型實跑記錄：FiftyOne 下載 10 筆，其中 9 張符合 `640×640` 條件。

## 4. 先檢查資料，再進行訓練

執行：

```powershell
python -m data.verify_kd_dataset --num-samples 10
```

驗證器預設使用 `data/raw/open-images-v6/train/data`，建立 deterministic validation Dataset，並對可用圖片逐張生成樣本，最多檢查指定的 10 筆。它會檢查：

- Teacher input 是否為 `[8, 6, 128, 128]`。
- Student reference 和 clean target 是否各為 `[3, 128, 128]`。
- sigma map 是否為 `[8, 3, 128, 128]`。
- 張量內容是否為 finite values。
- Teacher 的 frame 0 RGB 是否等於 Student reference。
- Teacher 的 3 個 sigma channels 是否等於 Dataset 回傳的 sigma map。

成功輸出 `KD Dataset generation and tensor checks passed.` 表示格式與生成路徑可用，不代表去噪品質，也不代表 noise distribution 已和預訓練 Teacher 完整匹配。本次 9 筆可用資料通過上述檢查。

## 5. Dataset sample 欄位說明

Dataset 在加入 DataLoader batch dimension 前回傳：

| Key | Shape / Type | 意義與使用位置 |
|---|---|---|
| `teacher_input` | float tensor `[8, 6, 128, 128]` | RGB noisy burst 與 sigma map 沿 channel axis 串接，送 Burstormer |
| `reference_rgb` | float tensor `[3, 128, 128]` | noisy frame 0，送 NAFNet |
| `synthetic_target` | float tensor `[3, 128, 128]` | clean frame 0，作 Ground-truth loss 的 target |
| `sigma_map` | float tensor `[8, 3, 128, 128]` | 每幀、每通道 noise standard deviation estimate |
| `white_level` | scalar float tensor | 該樣本的強度縮放參數 |
| `sigma_read` | scalar float tensor | 該樣本的 read-noise parameter |
| `sigma_shot` | scalar float tensor | 該樣本的 shot-noise parameter |
| `sample_seed` | scalar integer tensor | 該次合成所用 seed（train mode 取樣 seed） |
| `image_path` | string | 來源圖路徑，方便追溯 |

DataLoader 加入 batch dimension 後，Teacher input 為 `[1, 8, 6, 128, 128]`；Teacher 限制 batch size 1。

## 6. Noise synthesis 與 sigma map：已知事實及需再確認事項

目前 KD Dataset 的 burst/noise 邏輯參考組員提供的 `NAFNet (from gpmate)\burst_utils.py`，實際的 GPMate source folder 不會在 runtime 被 import。Burst generator 以 640×640 區域製作 8 個 128×128 frame；第 0 frame 是 reference。Noise generator 對一筆 sample 隨機取樣：

- `white_level = 10 ** uniform(-1, 0)`
- `sigma_read = 10 ** uniform(-3, -1.5)`
- `sigma_shot = 10 ** uniform(-2, -1)`

其 clean linear-like frame 由 `white_level * RGB ** 2.2` 得到；read noise 與依賴像素強度的 shot noise 分別加入。sigma map 形式是：

```text
sigma_map = sqrt(sigma_read² + sigma_shot² × max(noisy_pixel, 0))
```

這份 map 使用生成同一 noisy burst 時的參數，故和**本程式合成噪聲**保持數學一致。這一點本身仍不能證明其 noise level、gamma/linear domain、white-level scaling 等於 Burstormer pretrained model 的訓練分佈；需與組員核對模型訓練和官方 testing preprocessing，並保留測試方案。不要因 smoke test 成功就宣稱這些已驗證。

文件引用的論文「四組 noise settings」描述的是其測試 variants；不應無證據把它當成目前 GPMate generator 的訓練採樣方式。現行 GPMate generator 使用連續 log-uniform random values，專案 Dataset 沿用此設定。

## 7. 20K 的計數單位與現行程式限制

需要區分：

1. **source images**：下載的 Open Images 照片。
2. **synthetic burst sample/patch**：從 source crop 出一組 burst、noisy frames 與 clean reference。
3. **training samples**：從所有 usable samples 扣除 validation/test split 後，實際送入 optimizer 的資料。

論文提到的 20K synthetic burst patches 是 patch/sample 數的描述；它不自動代表需要下載 20K source photographs。現行 `NAFNetKDBurstDataset.__len__()` 等於符合圖片尺寸條件的 source images 數；每次 access 產生一組 sample。Training mode 每次 access 重抽 crop 和 noise，validation mode 使用 `seed + index`，因而同一來源圖在不同 epoch 可重抽不同 train sample。

`train_kd.py` 目前以圖片 index 進行 80/20 train/validation split，batch size 1、預設 50 epochs。假設恰有 20,000 張可用圖片，split 約為 16,000 train、4,000 validation；下載上限 20,000 張不保證實際有 20,000 張 usable images。

目前沒有固定 20K generated sample manifest、指定 epoch step 數的 sampler、或 pre-generation/cache script。正式訓練前必須和組員明確決定：

- 20K 是 train-only，還是包含 validation 的總 patches？
- 20K 是原圖數還是 burst patch 數？
- 是否允許同一 source image 反覆取不同 crops/noise？
- NAFNet baseline 與 KD 是否須逐筆完全相同的 crop/noise？目前只共用 generator 邏輯，不保證每筆隨機 draw 一樣。
- 如何切 train/validation/test，以及 test noise settings 如何定義？

如需逐筆公平對照，建議先定義共享 sample manifest（source image identity、crop seed、noise seed、parameters、reference frame），讓 baseline 和 KD 都從同一資料定義生成；這一能力尚未實作。

## 8. Smoke Dataset 和舊 `.npz` Dataset 的責任邊界

### `synthetic_smoke_dataset.py`

提供小型隨機資料，含常數 placeholder sigma channels 和隨機 toy target，供 `run_kd_smoke.py` 試跑 Teacher/Student shape、loss 和 optimizer update。它不是 Open Images loader，也不讀取正式噪聲參數，不可用它訓練或評分。

### `burst_dataset.py`

讀取含 `truth`、`noisy`、`sig_read`、`sqrt_sig_shot`、`white_level` 等鍵的 `.npz`，轉成 PyTorch layout、計算 sigma map，建立 Teacher input。此 loader 是先前預想的一種外部預製資料介面；目前的 `train_kd.py` 沒有 import 它，也不要求下載或製作 `.npz`。只有在 `.npz` producer 的 shape、shot parameter 是 sigma 還是 sqrt(sigma)、影像值域、frame order 和 white-level semantics 都核實後，才可接回正式 pipeline。

## 9. 交接 ZIP 和磁碟清理

交付純程式 ZIP 時，請省略 `data/raw/`：目前 10 張小樣本下載加 FiftyOne metadata 約 612 MB，組員可以在自己的環境重跑 downloader。`data/processed/` 目前空白且未被 training 使用，亦不必放進 ZIP。請保留 `data/` 內的 `.py` 和 README。

要清除目前 raw download，可在專案根目錄執行：

```powershell
Remove-Item -Recurse -Force .\data\raw
```

此命令會移除下載圖片和 FiftyOne metadata；下次要訓練須重新下載。請確認目前 PowerShell 位於 `3rd_KD_Completed`，再執行。**不要刪除整個 `data/`**，否則會刪除程式和文件。ZIP 清理項目亦包括 `.venv/`、`checkpoints_kd/`、Python cache，詳見根目錄 README。

## 10. 本模組目前狀態

**已完成**：下載器、小樣本下載、usable image 過濾、KD Dataset 即時資料合成、sigma map 綁定生成參數、sample contract 驗證，並完成 1 epoch CPU 小型 KD run。

**尚待完成／確認**：正式 20K 定義和 sample scheduling、baseline 與 KD 逐樣本對齊、noise/value-range 與 Teacher 預訓練相容性、獨立 test dataset、評分和 post-processing。

## 11. 核心實作細節（Implementation Walk-through）

### 11.1 來源圖轉換與 burst crop

Dataset 用 Pillow 開啟圖片並轉 RGB，轉為 float32，再除以 255。短邊需至少 640 pixel，因 generator 要先抽取 640×640 region。8 frames 共享同一個大區域；第 0 frame 固定使用中心起點，其他 frames 依 motion rule 產生不同 crop。每個 frame 將 512×512 crop 以 4×4 block mean downsample 成 128×128 RGB frame。

Burst frames 是從同一張 Open Images 靜態照片裁切產生，不是真正相機連拍的 8 張連續曝光。合成運動／位移只模擬 burst frame 間的視野差異；若研究要求真實 burst motion、alignment 或 sensor pipeline，需要替換或擴充這個資料來源，不能把目前合成 burst 說成真實連拍資料。

### 11.2 Noise generation 與輸出值域

對每筆 clean burst 取樣一個 white level，再做 `white_level × clean_rgb**2.2` 轉換。每個 burst 共用該 sample 的 `sigma_read` 和 `sigma_shot` 參數；read Gaussian noise 與乘上 `sqrt(clean_linear)` 的 shot Gaussian noise 加到所有 frames。實際程式回傳的是 noisy linear-like RGB，可能略低於 0 或超過預期上限；這些值域和 Burstormer checkpoint 的預訓練資料相容性尚未獨立驗證。

Noise map 在 noisy burst 生成後計算：

```python
sigma_map = sqrt(sigma_read**2 + sigma_shot**2 * maximum(noisy_burst, 0))
```

Clamp 在計算 variance 時只處理 shot-noise 的像素依賴項，不會把 noisy input 本身 clamp。Sigma map 的三個通道對應 RGB 每個 channel，各 frame 使用該 frame 的像素值。這保留了 generator 中實際使用的 noise parameters，避免先生成一批 noisy data 卻另外使用無關固定 sigma。

### 11.3 Reference 和 GT 對齊

`generate_clean_burst` 將第一張 clean burst frame 作為 reference。Noise generator 對整個 burst 加噪；clean target 由 `clean_linear[0].copy()` 取得。Dataset 回傳 noisy frame 0 給 Student、完整 burst 給 Teacher、clean frame 0 作 supervised target。因此兩模型的預測都對應 frame 0。若未來將 reference 改成中心 frame、重排 frames 或改用別的 target，應同時調整 Teacher 的參考設定及配對測試。

## 12. Dataset 的隨機性、重現性和多 worker

Validation mode 用 `seed + index` 初始化 NumPy Generator，同一個 index 可重現相同 crop 和 noise（前提是同一份 source images 順序與 generator code）。Training mode 在每次 `__getitem__` 用 PyTorch 產生新 seed，再以 NumPy Generator 抽 crop、motion、white-level 和 noise；故同一張 source image 在不同 epoch 通常會得到不同 sample。

目前回傳 `sample_seed` 和 `image_path`，可記錄一次 access 的來源與 seed；但 `train_kd.py` 沒有保存完整 epoch-by-epoch sample manifest，沒有保證 baseline 和 KD 的 random number stream 相同，也沒有提供精確重現已跑過 epoch 的完整追蹤機制。多 worker 的 random seed 行為也需在正式採用 `--num-workers > 0` 前測試，確保各 worker 不意外重複樣本。

若團隊要嚴格逐筆公平比較，建議之後把 sample manifest 設計成明確資料契約，例如每筆記錄 source image id、split、crop seed、motion seed、noise seed、white level、sigma parameters 和 reference index。Baseline 與 KD 兩條 pipeline 讀同一 manifest 才能確認生成結果一致；僅設同一 global seed 不一定代表兩個不同程式消耗 RNG 的順序完全一致。

## 13. 20K 與 Epoch 應如何描述

現有 Dataset 是 map-style Dataset，`__len__` 回傳 usable source images 數。每個 epoch 的 sampler 會遍歷 train split index 一次；每次讀取 train Dataset 時才重新合成 crop/noise。因為圖片對應的 crop 是隨機的，每個 epoch 的內容可以變，但每個 epoch 的 index 數不會因為即時重抽而自動增加。

舉例：若下載上限 20,000，但只有 19,000 張符合尺寸；80/20 split 約 15,200 train、3,800 validation。這是現行程式自然產生的計數，不是指定要做的研究規模。若論文 protocol 要求固定 20K train patches，應明確實作 `samples_per_epoch`/repeat sampler 或建立固定 manifest，而不是僅提高 `--max-samples`。若 20K 包含所有 split，也需把 validation/test 的數目從總量扣除。

另一方面，資料來源不足時可以重複抽取同一原圖的不同 crop/noise，但這會改變「20K」含義，也需考慮 train/validation image leakage。較安全的比較方式是先按 source image identity 切 split，再只在各 split 內生成 crop，且保存每一筆 sample 的可重現設定。當前程式以不同 index 做 image-level split，但不負責建立獨立 test split。

## 14. 如何檢查生成結果（Manual Inspection）

`verify_kd_dataset.py` 目前做自動張量檢查，沒有輸出可視化圖片。若要進一步目視確認，建議另寫不影響 training 的 preview 工具，將 `clean_target`、`reference_rgb` 和 `teacher_input[:, :3]` 轉回可顯示範圍；不同 `white_level`、gamma／display conversion 要正確處理。不要直接把線性 noisy tensor 當成一般 sRGB 圖顯示後，依顯示外觀判定噪聲參數正確。

人工確認可回答「frame 0 是否與 target 對齊、noise 是否可見、burst crop 是否合理」；不能替代 numeric tests、noise distribution check 或正式 Teacher output evaluation。

## 15. Download 命令與重新下載

小型測試：

```powershell
python -m data.download_open_images --max-samples 10 --seed 4008
python -m data.verify_kd_dataset --num-samples 10 --seed 4008
```

Downloader 預設 dataset name 由數量組成，例如 `oi_v6_train_10`。FiftyOne 會檢查磁碟上的 split 是否已足夠；重跑相同設定可能重用既有 Open Images images，而不是重新下載。若要指定新的 FiftyOne dataset name 可用 `--dataset-name`。不要為了重拉資料而直接刪整個 `data/`；若必須清除，先確認只刪除 `data/raw/` 會符合需求，並注意該目錄也含 metadata。

擴大下載前先確認可用磁碟空間。小型 run 中 raw tree 約 612 MB，其中大部分為 Open Images metadata，而圖片本體約數 MB；大規模圖片下載與官方 metadata 所需空間會顯著增加，不可把小樣本大小直接線性當成精確預算。

## 16. 新成員拿到 ZIP 後的 Data Checklist

1. 確認在專案根目錄執行，不是在 `data/` 裡執行 module command。
2. 建立 Python 環境、安裝 PyTorch 和 `requirements.txt`。
3. 確認下載程式可匯入 FiftyOne 並能連線取得 Open Images。
4. 執行 downloader；檢查回報的 Image files 和 usable count。
5. 執行 Dataset verifier；確認少量資料 tensors shape 和 finite check 通過。
6. 記錄 downloader seed、dataset name、來源 split、可用數量和程式版本。
7. 再執行小型 `train_kd.py`，確認 Teacher checkpoint path、device 和輸出目錄。
8. 正式訓練前與組員確認 sample count、noise protocol、split 和公平比較定義。

若資料下載正常，但 verifier 回報 `No readable images ... at least 640x640`，代表目前下載的圖片都不符合 crop 條件；增加下載上限或換來源資料，不要自行降低尺寸門檻，除非同步修改並驗證 burst generator。
